# 02 - Product troubleshooting: near-duplicates, error codes and corrective RAG

**Scenario.** A support executive is helping a customer whose **Voltara Lumina 55 Pro** shows error **E-715** when a
game console is connected. The catalogue has near-duplicate manuals (Lumina 55 vs Lumina 55 **Pro**), and error codes
are exactly the kind of token embeddings blur.

**You will see**
1. Similarity search without and with a **metadata filter** (model pinning).
2. **BM25** keyword search vs vector search for the code `E-715`.
3. **Hybrid** search with Reciprocal Rank Fusion (which retriever found what).
4. **Corrective RAG** on a question the manuals cannot answer.
5. The **Product Manuals agent** (a Deep Agent) producing a structured report.

**Prerequisites:** `.env` with `OPENAI_API_KEY`, and the vector stores (`make ingest`).

In [1]:
# Setup: make src/ importable and load the settings from .env
import sys, warnings
from pathlib import Path

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore", message=".*langchain.mcp.*")

from agentic_rag.config import AgentSettings

settings = AgentSettings.load(ROOT)
print(f"Coordinator: {settings.model} ({settings.coordinator_effort}) | specialists: {settings.specialist_model} "
      f"({settings.specialist_effort}) | judge: {settings.judge_model} (pass >= {settings.judge_pass_score})")

Coordinator: gpt-5.1 (high) | specialists: gpt-5.1 (low) | judge: gpt-5.5 (pass >= 80)


## 1. Near-duplicate manuals: pin the model

In [2]:
from agentic_rag.retrieval import MetadataFilter, infer_filter, similarity_search, keyword_search, hybrid_search

q = "What is the native refresh rate of the Lumina 55?"
print("No filter:")
for c in similarity_search(settings, "products", q, 5):
    print(f"  {c.score:.3f} {c.citation():<18} model={c.metadata.get('model')}")
flt = infer_filter(settings, "products", q)
print("Inferred filter:", flt.constraints())
for c in similarity_search(settings, "products", q, 5, flt):
    print(f"  {c.score:.3f} {c.citation():<18} model={c.metadata.get('model')}")

No filter:
  0.568 TV-LUM-Q55 p.8     model=QLED Q55
  0.566 TV-VOL-L43 p.9     model=Lumina 43
  0.559 TV-LUM-Q55X p.8    model=QLED Q55X
  0.552 TV-LUM-Q55 p.6     model=QLED Q55
  0.547 TV-LUM-Q55X p.8    model=QLED Q55X
Inferred filter: {'model': 'Lumina 55'}
  0.524 TV-VOL-L55 p.9     model=Lumina 55
  0.513 TV-VOL-L55 p.8     model=Lumina 55
  0.492 TV-VOL-L55 p.5     model=Lumina 55
  0.467 TV-VOL-L55 p.3     model=Lumina 55
  0.439 TV-VOL-L55 p.1     model=Lumina 55


## 2. Error codes: BM25 vs vectors

In [3]:
for name, hits in [("vector", similarity_search(settings, "products", "E-715", 5)),
                   ("bm25", keyword_search(settings, "products", "E-715", 5))]:
    print(name)
    for c in hits:
        snippet = c.text[c.text.find("E-715"):][:70].replace("\n", " ") if "E-715" in c.text else "(no E-715 in chunk)"
        print(f"  {c.citation():<18} {c.metadata.get('model'):<14} {snippet}")

vector
  TV-VOL-L55P p.9    Lumina 55 Pro  (no E-715 in chunk)
  TV-VOL-L55P p.7    Lumina 55 Pro  E-715 or E-730. If you need to contact Voltara customer care or Voltar
  TV-VOL-L55 p.8     Lumina 55      (no E-715 in chunk)
  TV-LUM-Q55 p.1     QLED Q55       (no E-715 in chunk)
  TV-VOL-L43 p.1     Lumina 43      (no E-715 in chunk)
bm25
  TV-VOL-L55P p.7    Lumina 55 Pro  E-715 or E-730. If you need to contact Voltara customer care or Voltar
  TV-VOL-L55P p.8    Lumina 55 Pro  E-715 HDMI 2.1 FRL link training failed. Use a certified Ultra High Sp
  TV-VOL-L65P p.8    Lumina 65 Pro  E-715 HDMI 2.1 FRL link training failed. Use a certified Ultra High Sp
  TV-VOL-L55 p.8     Lumina 55      (no E-715 in chunk)
  TV-VOL-L43 p.8     Lumina 43      (no E-715 in chunk)


## 3. Hybrid search with RRF
`ranks` shows the position of each chunk in the dense and the BM25 list.

In [4]:
for c in hybrid_search(settings, "products", "What does error E-715 mean?", k=5,
                       flt=MetadataFilter(model="Lumina 55 Pro")):
    print(f"{c.score:.5f} {c.label():<45} ranks={c.ranks}")

0.03279 TV-VOL-L55P p.7 | Error Codes | v1.2          ranks={'vector': 1, 'bm25': 1}
0.03200 TV-VOL-L55P p.8 | Error Codes | v1.2          ranks={'vector': 3, 'bm25': 2}
0.03151 TV-VOL-L55P p.1 | Front matter | v1.2         ranks={'vector': 2, 'bm25': 5}
0.03150 TV-VOL-L55P p.7 | Troubleshooting | v1.2      ranks={'vector': 4, 'bm25': 3}
0.03101 TV-VOL-L55P p.9 | Regulatory and Environmental Information | v1.2 ranks={'vector': 5, 'bm25': 4}


## 4. Corrective RAG: grade, broaden, switch store, admit "I don't know"

In [5]:
from agentic_rag.rag import corrective_rag

for question in ["How do I fix error E-715 on the Lumina 55 Pro?",
                 "What is the retail price of the Lumina 55 Pro in Singapore?"]:
    r = corrective_rag(settings, "products", question)
    print("Q:", question)
    print("\n".join("   " + t for t in r.trace))
    print("A:", r.answer, "\n")

Q: How do I fix error E-715 on the Lumina 55 Pro?
   rewrite: 'How do I fix error E-715 on the Voltara Lumina 55 Pro TV? troubleshooting steps, error codes section, reset instructions'
   hybrid candidates: 20 (filter: {'model': 'Lumina 55 Pro'})
   rerank: TV-VOL-L55P p.8=3, TV-VOL-L55P p.7=2, TV-VOL-L55P p.7=1, TV-VOL-L55P p.8=1, TV-VOL-L55P p.7=0, TV-VOL-L55P p.2=0
   grade (first pass): 1/6 relevant
   correct: too little relevant evidence -> original wording, no filters, MMR retrieval
   grade (broadened): 1/5 relevant
   context: 2 sources, 505 tokens, 0 dropped
A: Use a certified Ultra High Speed HDMI cable and connect it to an HDMI 2.1 port on the Lumina 55 Pro to resolve error E‑715 (HDMI 2.1 FRL link training failed). [S2] 

Q: What is the retail price of the Lumina 55 Pro in Singapore?
   rewrite: 'retail price of Voltara Lumina 55 Pro TV in Singapore'
   hybrid candidates: 20 (filter: {'model': 'Lumina 55 Pro'})
   rerank: TV-VOL-L55P p.9=3, TV-VOL-L55P p.3=0, TV-VOL-L55P p

## 5. The Product Manuals agent
A Deep Agent with only the four RAG tools on the products store and the `product-support` + `rag-answering` skills. It returns an `AgentReport`, never free text.

In [6]:
from agentic_rag.agents import product_manuals
from agentic_rag.agents.base import build_specialist, run_specialist, task_message

agent = build_specialist(product_manuals.SPEC, settings, product_manuals.tools(settings))
goal = "Explain error E-715 on the Voltara Lumina 55 Pro and list the fix steps from the manual."
report = await run_specialist(agent, "product_manuals", "s1", goal, task_message(goal, "s1", goal))
print(f"confidence={report.confidence} duration={report.duration_s}s")
print(report.findings)
for e in report.evidence:
    print(" -", e.cite(), "|", e.quote[:100])
print("gaps:", report.gaps)

confidence=0.97 duration=8.6s
On the Voltara Lumina 55 Pro (model VT-55L7Q), error E-715 is defined in the manual as an HDMI 2.1 FRL link training failure. The manual’s recommended fix is to use a certified Ultra High Speed HDMI cable and connect the device to an HDMI 2.1-capable port on the TV. The error appears in the error code list for this specific model, which maps each code to its meaning and corrective action.
 - TV-VOL-L55P p.8 | E-715 HDMI 2.1 FRL link training failed. Use a certified Ultra High Speed HDMI cable and connect to 
 - TV-VOL-L55P p.7 | Error Codes When the TV detects a problem, it shows an error code beginning with E- in an on-screen 
gaps: []


## What to observe
- Without the filter, Lumina 55, 55 Pro and 65 Pro chunks compete; the filter keeps only the asked model.
- BM25 finds `E-715` literally; the vector search returns generally related chunks.
- Corrective RAG grades evidence, broadens and even tries the other store before saying "I don't know".
- The agent's report carries page-level evidence with verbatim quotes.